### NOTE:
###### This notebook was used for early exploration before the ,Silver layer was filtered to Europe + km-only events.Kept for documentation purposes; results reflect the original, unfiltered global dataset (km + miles, all countries).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

df = spark.read.table("marathos.bronze.races")

# Casta alla kolumner till string tillfälligt bara för null-räkning
df_str = df.select([F.col(c).cast(StringType()).alias(c) for c in df.columns])

null_counts = df_str.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in df_str.columns
]).toPandas().T.rename(columns={0: "nulls"})

null_counts["pct"] = (null_counts["nulls"] / df.count() * 100).round(1)
print(null_counts.sort_values("pct", ascending=False))

In [0]:
df.groupBy(
    F.regexp_extract("event_distance_length", r"[a-z]+$", 0).alias("suffix")
).count().orderBy(F.col("count").desc()).show()

In [0]:
# Hur många rader överlever event_unit-filtret?
from pyspark.sql import functions as F

df = spark.read.table("marathos.bronze.races")

df.withColumn(
    "event_unit",
    F.when(F.col("event_distance_length").endswith("km"), "km")
     .when(F.col("event_distance_length").endswith("mi"), "mi")
     .when(F.col("event_distance_length").endswith("h"),  "h")
     .when(F.col("event_distance_length").endswith("d"),  "d")
     .otherwise("SKRÄP")
).groupBy("event_unit").count().orderBy(F.col("count").desc()).show()

In [0]:
# Cell 3 – hastighetsspridning
df.select(
    F.regexp_replace("athlete_average_speed", "[^0-9.]", "").cast("float").alias("speed_num")
).summary("min", "25%", "75%", "max").show()

In [0]:
# Cell 4 – performance-format
df.groupBy(
    F.when(F.col("athlete_performance").contains(":"), "HH:MM:SS")
     .when(F.col("athlete_performance").rlike(r"^\d+\.\d+"), "decimal")
     .otherwise("övrigt")
     .alias("format")
).count().show()

In [0]:

# ═══════════════════════════════════════
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# ── UDFs ──────────────────────────────
def time_to_seconds(time_str):
    if time_str is None: return None
    try:
        s = time_str.strip()
        # Remove suffix like " h"
        s = s.split(" ")[0]
        extra_days = 0
        if "day" in s:
            parts = s.split(",")
            extra_days = int(parts[0].strip().split(" ")[0])
            s = parts[1].strip()
        hms = s.split(":")
        if len(hms) == 3:
            h, m, sec = int(hms[0]), int(hms[1]), int(float(hms[2]))
            return extra_days * 86400 + h * 3600 + m * 60 + sec
        return None
    except Exception:
        return None

time_to_seconds_udf = F.udf(time_to_seconds, IntegerType())

# ── STEP 1: Read Bronze ───────────────
df_bronze = spark.read.table("marathos.bronze.races")
print(f"Bronze rows: {df_bronze.count():,}")

# ── STEP 2: Extract units ─────────────
df_with_units = (
    df_bronze
    .withColumn(
        "event_unit",
        F.when(F.col("event_distance_length").endswith("km"), "km")
         .when(F.col("event_distance_length").endswith("mi"), "mi")
         .when(F.col("event_distance_length").endswith("h"),  "h")
         .when(F.col("event_distance_length").endswith("d"),  "d")
         .otherwise(None)
    )
    .withColumn(
        "performance_unit",
        F.when(F.col("athlete_performance").contains(":"), "h")
         .otherwise("km")
    )
    .withColumn(
        "is_valid_combo",
        F.when(
            (F.col("event_unit").isin("km", "mi")) & (F.col("performance_unit") == "h"), True
        ).when(
            (F.col("event_unit") == "h") & (F.col("performance_unit") == "km"), True
        ).otherwise(False)
    )
)

# ── STEP 3: Remove invalid rows ───────
df_clean = (
    df_with_units
    .filter(F.col("event_unit") != "d")
    .filter(F.col("is_valid_combo") == True)
    .filter(F.col("athlete_performance").isNotNull())
    .filter(F.col("event_distance_length").isNotNull())
    .filter(F.col("athlete_id").isNotNull())
    # Remove invalid speeds
    .filter(~F.col("athlete_average_speed").contains(":"))
    .filter(
        F.regexp_replace(F.col("athlete_average_speed"), "[^0-9.]", "").cast("float").between(0,5, 35)
        | F.col("athlete_average_speed").isNull()
    )
)
print(f"Clean rows: {df_clean.count():,}")

# ── STEP 4: Convert performance ───────
df_converted = (
    df_clean
    .withColumn(
        "athlete_performance_clean",
        F.trim(F.regexp_replace(F.col("athlete_performance"), r"\s*(h|km|mi)$", ""))
    )
    .withColumn(
        "performance_seconds",
        F.when(
            F.col("performance_unit") == "h",
            time_to_seconds_udf(F.col("athlete_performance_clean"))
        ).otherwise(None)
    )
    .withColumn(
        "performance_km",
        F.when(
            F.col("performance_unit") == "km",
            F.col("athlete_performance_clean").cast("float")
        ).otherwise(None)
    )
    .drop("athlete_performance_clean")
)

# ── STEP 5: Create IDs ────────────────
w_event   = Window.orderBy("event_name")
w_athlete = Window.orderBy("athlete_id")
w_year    = Window.orderBy("year_of_event")

df_ids = (
    df_converted
    .withColumn("event_id", F.dense_rank().over(w_event))
    .withColumn("athlete_key", F.dense_rank().over(w_athlete))
    .withColumn("year_id", F.dense_rank().over(w_year))
)

# ── STEP 6: Final cleanup ─────────────
df_silver = (
    df_ids
    .withColumn("athlete_average_speed",     F.expr("try_cast(split(athlete_average_speed, ' ')[0] as float)"))
    .withColumn("athlete_year_of_birth",     F.col("athlete_year_of_birth").cast("int"))
    .withColumn("event_number_of_finishers", F.col("event_number_of_finishers").cast("int"))
    .withColumn("athlete_gender",  F.upper(F.trim(F.col("athlete_gender"))))
    .withColumn("athlete_country", F.upper(F.trim(F.col("athlete_country"))))
    .withColumn("athlete_age_at_event", F.col("year_of_event") - F.col("athlete_year_of_birth"))
    .drop("is_valid_combo", "event_unit", "performance_unit")
)

print(f"Bronze rows : {df_bronze.count():,}")
print(f"Clean rows  : {df_clean.count():,}")
print(f"Silver rows : {df_silver.count():,}")

removed = df_bronze.count() - df_silver.count()

print(f"Removed rows: {removed:,}")

print(
    f"Removed percentage: {removed / df_bronze.count() * 100:.2f}%"
)

df_silver.printSchema()
display(df_silver.limit(20))

In [0]:
# Null analysis
display(
    df_silver.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_silver.columns
    ])
)

In [0]:
# Age distribution
display(
    df_silver.groupBy("athlete_age_at_event")
             .count()
             .orderBy("athlete_age_at_event")
)

In [0]:
# Speed distribution
display(
    df_silver.select(
        F.min("athlete_average_speed"),
        F.max("athlete_average_speed"),
        F.avg("athlete_average_speed")
    )
)